# DeepShelf: Book Recommendation System Using Machine Learning


##1. Import Libraries

In [ ]:
# Import libraries used for data handling, modeling, and evaluation
import pandas as pd
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load and Prepare the Ratings Dataset

In [ ]:
# Load the ratings dataset
df = pd.read_csv('/content/drive/MyDrive/DeepShelf/BX-Book-Ratings.csv.zip', sep=';', encoding='latin-1')
# Rename columns to simpler names
df.columns = ['user_id', 'isbn', 'rating']

# Create a binary target variable
# A rating of 7 or higher is treated as "liked"
df['liked'] = (df['rating'] >= 7).astype(int)

# Reduce dataset size for faster processing
df = df.sample(25000, random_state=42)

# Show first rows
df.head()

,user_id,isbn,rating,liked
178554,38781,0373259131,0,0
533905,128835,0811805905,8,1
1091374,261829,037324486X,0,0
1036247,247747,0531303306,0,0
309523,74076,0316812404,0,0


## 3. Load and Prepare the Books Dataset

In [ ]:
# Load the books dataset
books = pd.read_csv('/content/drive/MyDrive/DeepShelf/Books.csv.zip', encoding='latin-1', on_bad_lines='skip')

# Keep only relevant columns
books = books[['ISBN', 'Book-Author', 'Year-Of-Publication']]

# Rename columns to match main dataset
books.columns = ['isbn', 'author', 'year']

# Show first rows
books.head()

/tmp/ipykernel_24805/1021648758.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('/content/drive/MyDrive/DeepShelf/Books.csv.zip', encoding='latin-1', on_bad_lines='skip')


,isbn,author,year
0,0195153448,Mark P. O. Morford,2002
1,0002005018,Richard Bruce Wright,2001
2,0060973129,Carlo D'Este,1991
3,0374157065,Gina Bari Kolata,1999
4,0393045218,E. J. W. Barber,1999


## 4. Merge Datasets and Clean Features

In [ ]:
# Merge the ratings dataset with the books dataset using the ISBN column
df = df.merge(books, on='isbn', how='left')

# Fill missing author values with "Unknown"
df['author'] = df['author'].fillna("Unknown")

# Convert year values to numeric
# Invalid values are turned into missing values
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Replace missing year values with 0
df['year'] = df['year'].fillna(0)

# Show the first few rows after merging and cleaning
df.head()

,user_id,isbn,rating,liked,author,year
0,38781,0373259131,0,0,Cara Summers,2001.0
1,128835,0811805905,8,1,Bruce Velick,1995.0
2,261829,037324486X,0,0,Lindsay Mckenna,2002.0
3,247747,0531303306,0,0,Ralph J. Fletcher,2001.0
4,74076,0316812404,0,0,Gloria Steinem,1992.0


## 5. Split Data and Create Stronger Features

I created user and book-based feature such as average ratings and counts to capture bevahioral patterns.

In [ ]:
# Select the base features and the target variable
X_base = df[['user_id', 'isbn', 'author', 'year']].copy()
y = df['liked']

# Split the data into training and testing sets
# This is done before feature engineering to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

# Create a temporary training table that includes the original rating values
train_stats = X_train.copy()
train_stats['rating'] = df.loc[X_train.index, 'rating']

# Create stronger features from training data only
user_avg = train_stats.groupby('user_id')['rating'].mean()
user_count = train_stats.groupby('user_id')['rating'].count()
book_avg = train_stats.groupby('isbn')['rating'].mean()
book_count = train_stats.groupby('isbn')['rating'].count()

# Add the engineered features to the training set
X_train['user_avg'] = X_train['user_id'].map(user_avg)
X_train['user_count'] = X_train['user_id'].map(user_count)
X_train['book_avg'] = X_train['isbn'].map(book_avg)
X_train['book_count'] = X_train['isbn'].map(book_count)

# Add the engineered features to the test set
X_test['user_avg'] = X_test['user_id'].map(user_avg)
X_test['user_count'] = X_test['user_id'].map(user_count)
X_test['book_avg'] = X_test['isbn'].map(book_avg)
X_test['book_count'] = X_test['isbn'].map(book_count)

# Fill missing values for unseen users or books
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Show the first few rows of the training data
X_train.head()

,user_id,isbn,author,year,user_avg,user_count,book_avg,book_count
23311,248718,0679738460,Howard M. Sachar,1995.0,7.086957,23,10.000000,1
23623,236757,0345329031,Robert Shea,1986.0,1.000000,10,0.000000,1
1020,13273,0393317552,Jared Diamond,1999.0,0.000000,7,3.333333,3
12645,106550,0671687948,John Irving,1989.0,7.000000,1,7.000000,1
1533,42914,0345313860,ANNE RICE,1986.0,2.000000,10,1.142857,7


## 6. Encode Categorical Features

In [ ]:
# Convert categorical text columns into numeric codes
# This allows the model to use them as input features

# Encode isbn and author in the training set
X_train['isbn'] = X_train['isbn'].astype('category').cat.codes
X_train['author'] = X_train['author'].astype('category').cat.codes

# Encode isbn and author in the test set
X_test['isbn'] = X_test['isbn'].astype('category').cat.codes
X_test['author'] = X_test['author'].astype('category').cat.codes

# Show the first rows after encoding
X_train.head()

,user_id,isbn,author,year,user_avg,user_count,book_avg,book_count
23311,248718,10275,3037,1995.0,7.086957,23,10.000000,1
23623,236757,2706,6722,1986.0,1.000000,10,0.000000,1
1020,13273,4780,3505,1999.0,0.000000,7,3.333333,3
12645,106550,9798,3898,1989.0,7.000000,1,7.000000,1
1533,42914,2665,64,1986.0,2.000000,10,1.142857,7


## 7. Train the Random Forest Model

A Random Forest model was used due to its ability to handle structured data and capture non-linear relationships.

In [ ]:
# Create the Random Forest model
# This model predicts whether a user will like a book
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train the model using the training data
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, random_state=42)

## 8. Evaluate Model Performance

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Display evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6984

Confusion Matrix:
 [[3419  128]
 [1380   73]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.96      0.82      3547
           1       0.36      0.05      0.09      1453

    accuracy                           0.70      5000
   macro avg       0.54      0.51      0.45      5000
weighted avg       0.61      0.70      0.61      5000



## 9. Threshold Tuning
Threshold was adjusted to improve recall for liked books and balance model performance.

In [ ]:
# Get probabilities for class 1 ("liked")
y_probs = model.predict_proba(X_test)[:, 1]

# Apply a lower threshold
threshold = 0.1
y_pred_adjusted = (y_probs >= threshold).astype(int)

# Evaluate the new predictions
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_adjusted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_adjusted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_adjusted))

Threshold: 0.1
Accuracy: 0.6846

Confusion Matrix:
 [[3127  420]
 [1157  296]]

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.88      0.80      3547
           1       0.41      0.20      0.27      1453

    accuracy                           0.68      5000
   macro avg       0.57      0.54      0.54      5000
weighted avg       0.64      0.68      0.65      5000



## 9. Final Observation
The final model achieved moderate performance in predicting whether a user would like a book. The initial model showed high accuracy but very low recall for liked books, meaning it failed to correctly identify many positive cases. To address this, threshold tuning was applied, which improved the model’s ability to detect liked books by increasing recall, although it slightly reduced overall accuracy.

This tradeoff highlights an important aspect of machine learning models: improving one metric often impacts others. In this project, improving recall was important to make the recommendation system more useful, since correctly identifying books that users may like is more valuable than simply maximizing accuracy.

Despite these improvements, the model still shows limitations due to the dataset. The available features do not include detailed information such as book genres, descriptions, or deeper user preferences. As a result, the model relies mainly on statistical patterns like average ratings and interaction counts, which limits its ability to fully capture user interests.

Overall, the model demonstrates a working recommendation approach and shows how feature engineering and threshold tuning can improve performance, while also highlighting the importance of richer data for building more accurate recommendation systems.